# Multi-Agent Reinforcement Learning: Everything you need to know

Welcome to the cutting edge of AI. In standard Reinforcement Learning, we define a **Markov Decision Process (MDP)** with a state space $S$, an action space $A$, a transition function $P$, and a reward function $R$.

In MARL, we upgrade this to a **Markov Game** (or Stochastic Game). The environment now has $N$ agents.
* The state is still $S$.
* The joint action space is $\mathbf{A} = A_1 \times A_2 \dots \times A_N$.
* The transition function $P(s' | s, \mathbf{a})$ depends on the joint actions of *all* agents.
* Every agent $i$ receives its own reward $R_i(s, \mathbf{a})$.

### The Non-Stationarity Problem
The hardest mathematical challenge in MARL is **Non-Stationarity**. From Agent 1's perspective, Agent 2 is just part of the environment. As Agent 2 learns and changes its behavior, the environment's transition dynamics appear to shift arbitrarily, destroying Agent 1's mathematical assumptions and preventing convergence.

### Our Analytical Pipeline:
1. **Environment Setup:** Instantiate a cooperative PettingZoo environment.
2. **The AEC API:** Understand the Agent-Environment-Cycle.
3. **Parameter Sharing:** Solve non-stationarity by forcing agents to share a single brain.
4. **Training:** Train the agents using Proximal Policy Optimization (PPO).

In [ ]:
# Cell 2: Environment Setup and Imports
# In a real environment, run: !pip install -q pettingzoo supersuit stable-baselines3
import pettingzoo
from pettingzoo.mpe import simple_spread_v3
import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy
import numpy as np

print(f"PettingZoo Version: {pettingzoo.__version__}")

## Step 1: The PettingZoo AEC API

In standard single-agent RL, the environment takes an action and returns `(state, reward, done, info)`. 

In MARL, if agents act simultaneously, applying this is tricky. PettingZoo uses the **AEC (Agent Environment Cycle)** API. It forces environments to step sequentially (Agent 1 acts, then Agent 2 acts, then the environment updates). This prevents race conditions and makes board games (like Chess) and physics environments share the exact same logical API.

Let's initialize the `simple_spread_v3` environment. Here, $N$ agents must learn to cover $N$ landmarks without colliding.

In [ ]:
# Cell 4: Initializing the Environment and Exploring the API

# Instantiate the environment with 3 agents
env = simple_spread_v3.env(N=3, local_ratio=0.5, max_cycles=25)

# Reset the environment to begin
env.reset()

print(f"Agents in environment: {env.agents}")

# Let's analytically examine the observation and action spaces for the first agent
first_agent = env.agents[0]
obs_space = env.observation_space(first_agent)
act_space = env.action_space(first_agent)

print(f"Observation Space ({first_agent}): {obs_space}")
# Observation is an array of physical data: [self_vel, self_pos, landmark_rel_positions, other_agent_rel_positions, communication]
print(f"Action Space ({first_agent}): {act_space}")
# Action is Discrete(5): [no_move, right, left, up, down]

## Step 2: Running a Random Policy in AEC

To truly understand how MARL works, we must iterate through the AEC loop. 

Instead of a single `env.step()`, we use `env.agent_iter()`. This iterator mathematically guarantees that we evaluate agents in the correct order, handling dead agents and environment updates perfectly under the hood.

In [ ]:
# Cell 6: The AEC Loop with Random Agents

env.reset()
total_rewards = {agent: 0 for agent in env.agents}

# env.agent_iter() is a generator that yields the name of the agent whose turn it is
for agent in env.agent_iter():
    
    # Observe the current state of the environment for THIS specific agent
    observation, reward, termination, truncation, info = env.last()
    
    total_rewards[agent] += reward
    
    if termination or truncation:
        action = None # The agent is dead or the game is over, pass a null action
    else:
        # Sample a completely random mathematical action
        action = env.action_space(agent).sample()
        
    # Step the environment forward with the chosen action
    env.step(action)

print("--- Random Policy Results (1 Episode) ---")
for agent, reward in total_rewards.items():
    print(f"{agent} Total Reward: {reward:.2f}")

print("\nAnalytical Observation: The rewards are highly negative because the agents are moving randomly, failing to cover landmarks, and colliding with each other!")

## Step 3: Parameter Sharing via SuperSuit

How do we train these 3 agents? We could create 3 separate PPO neural networks (Independent Learning). However, as discussed, this leads to non-stationarity and takes 3x the memory.

Since our agents are homogeneous (they have the exact same action/observation spaces and identical goals), we use **Parameter Sharing**. 
We train a *single* Neural Network. We pass Agent 1's observation in, get an action, then pass Agent 2's observation into the *same* network, and get an action. 

This gives the network 3x the experience per step and elegantly solves non-stationarity because the "other" agents' logic is mathematically tied to the current agent's logic.

We use **SuperSuit** to wrap our PettingZoo environment so `stable_baselines3` thinks it is interacting with just one massive, vectorized single-agent environment.

In [ ]:
# Cell 8: Vectorizing and Wrapping the Environment

# 1. We must use the 'parallel' API for Stable Baselines 3, which executes actions simultaneously
parallel_env = simple_spread_v3.parallel_env(N=3, local_ratio=0.5, max_cycles=25)

# 2. Convert observations to float32 (Standard for PyTorch neural networks)
parallel_env = ss.dtype_v0(parallel_env, 'float32')

# 3. Concatenate the parallel env into a single vectorized environment.
# This is the Parameter Sharing magic trick. SB3 will see this as 3 parallel single-agent environments.
vec_env = ss.pettingzoo_env_to_vec_env_v1(parallel_env)
vec_env = ss.concat_vec_envs_v1(vec_env, num_vec_envs=1, num_cpus=1, base_class='stable_baselines3')

print("Environment successfully wrapped and vectorized for Stable-Baselines3!")

## Step 4: Training with PPO

Now that our multi-agent environment is mathematically identical to a single-agent vectorized environment, we can simply plug in standard **Proximal Policy Optimization (PPO)**.

PPO uses a clipping function in its objective to ensure that the neural network weights do not update too drastically during training, providing excellent stability—which is doubly important in MARL.

$$L^{CLIP}(\theta) = \hat{\mathbb{E}}_t \left[ \min(r_t(\theta)\hat{A}_t, \text{clip}(r_t(\theta), 1 - \epsilon, 1 + \epsilon)\hat{A}_t) \right]$$

In [ ]:
# Cell 10: Training the Shared Neural Network

# Instantiate the PPO model
# MlpPolicy creates a standard feed-forward neural network
model = PPO(
    MlpPolicy, 
    vec_env, 
    verbose=1, 
    learning_rate=3e-4, 
    batch_size=256
)

print("Starting Training (This would run for millions of timesteps in a real scenario)...")

# We train for a very small number of timesteps here for demonstration purposes
model.learn(total_timesteps=10_000)

print("Training Complete! The single PPO network has learned a shared policy for all 3 agents.")

## Conclusion

You have successfully navigated the architecture of Multi-Agent Reinforcement Learning!

**Key Analytical Takeaways:**
1. **The Game Theory Transition:** Moving from MDPs to Markov Games introduces other learning entities, destroying the assumption that the environment is static.
2. **The API Bridge:** PettingZoo's AEC API handles the strict turn-based logic required to simulate multi-agent physics without race conditions.
3. **The Parameter Sharing Trick:** By using `SuperSuit`, we flattened a 3-agent environment into a batched 1-agent environment. This allows us to use robust algorithms like PPO without dealing with the instability of 3 separate neural networks fighting each other.

To expand on this, the next logical step in MARL research is **CTDE (Centralized Training, Decentralized Execution)** algorithms like QMIX or MADDPG, where the agents have a "god-view" critic during training, but act entirely independently during deployment!